# Modul 1 — Preprocessing Teks (50 Dokumen STKI)

**Tujuan sesuai modul:** melakukan case folding, tokenisasi, stopword removal, stemming bahasa Indonesia (Sastrawi), serta membandingkan jumlah token dan kosakata di setiap tahap.

**Dataset aktual:** `../Data/korpus.csv` (50 dokumen), `../Data/queries.csv` (25 query), `../Data/qrels.csv` (59 pasangan penilaian awal dengan skor 1–3). Dokumen menggunakan kolom `id_doc`, `judul`, `isi`. Qrels merupakan label evaluasi dan **tidak dipakai untuk melatih preprocessing**.

Jalankan notebook dari folder `notebooks/`. Bila belum tersedia, instal dependensi: `pip install pandas numpy matplotlib Sastrawi`.


In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Langkah 1 — Memuat korpus

In [2]:
from pathlib import Path
DATA_DIR = Path("../Data")
if not DATA_DIR.exists():
    DATA_DIR = Path("Data")
korpus = pd.read_csv(DATA_DIR / "korpus.csv")
queries = pd.read_csv(DATA_DIR / "queries.csv")
qrels = pd.read_csv(DATA_DIR / "qrels.csv")
assert list(korpus.columns) == ["id_doc", "judul", "isi"]
assert korpus["id_doc"].is_unique and queries["query_id"].is_unique
assert set(qrels.doc_id).issubset(set(korpus.id_doc))
assert set(qrels.query_id).issubset(set(queries.query_id))
print(f"Korpus: {len(korpus)} dokumen | Query: {len(queries)} | Qrels: {len(qrels)} pasangan")
display(korpus.head(3), queries.head(3), qrels.head(3))


Korpus: 50 dokumen | Query: 25 | Qrels: 59 pasangan


,id_doc,judul,isi
0,D01,Pertanian,"""Pertanian mencakup kegiatan budidaya tanaman dan peternakan untuk menghasilkan pangan, bahan baku, dan produk lain...."
1,D02,Peran AI Generatif dalam Industri Kreatif,"Kemajuan teknologi kecerdasan buatan (AI) terus berkembang pesat dan memengaruhi berbagai sektor industri, termasuk ..."
2,D03,Peta,"Makna ""peta"" bergantung pada konteksnya.[4][a] Di dalam ranah kartografi, definisi awal berfokus pada representasi B..."


,query_id,query
0,Q01,Teknologi kecerdasan buatan untuk menghasilkan konten kreatif
1,Q02,Algoritma machine learning untuk prediksi data
2,Q03,Penyebab dan dampak kebakaran hutan dan lahan


,query_id,doc_id,relevance
0,Q01,D02,3
1,Q01,D23,3
2,Q01,D43,1


In [3]:
# Pilih contoh yang menggambarkan topik AI generatif (D02).
korpus["teks"] = korpus["judul"].fillna("") + ". " + korpus["isi"].fillna("")
contoh = korpus.loc[korpus["id_doc"] == "D02", "teks"].iloc[0]
print("Contoh D02:", contoh)


Contoh D02: Peran AI Generatif dalam Industri Kreatif. Kemajuan teknologi kecerdasan buatan (AI) terus berkembang pesat dan memengaruhi berbagai sektor industri, termasuk industri kreatif. AI generatif, yang menggunakan algoritma untuk menciptakan konten baru berdasarkan data yang ada, telah mengubah cara manusia berinteraksi dengan seni, musik, desain, dan bahkan film. Alat-alat seperti ChatGPT dan DALL-E dari OpenAI adalah contoh nyata dari AI generatif yang mampu menghasilkan teks dan gambar berdasarkan deskripsi singkat. Dampak teknologi ini semakin terasa dalam proses kreatif, mempercepat produktivitas dan membuka peluang baru bagi seniman dan kreator.


## Langkah 2 — Case folding
*Case folding* menyeragamkan huruf menjadi huruf kecil sehingga `Stemming`, `stemming`, dan `STEMMING` dianggap kata yang sama.

In [4]:
def case_folding(teks):
    return teks.lower()

print(case_folding(contoh))

peran ai generatif dalam industri kreatif. kemajuan teknologi kecerdasan buatan (ai) terus berkembang pesat dan memengaruhi berbagai sektor industri, termasuk industri kreatif. ai generatif, yang menggunakan algoritma untuk menciptakan konten baru berdasarkan data yang ada, telah mengubah cara manusia berinteraksi dengan seni, musik, desain, dan bahkan film. alat-alat seperti chatgpt dan dall-e dari openai adalah contoh nyata dari ai generatif yang mampu menghasilkan teks dan gambar berdasarkan deskripsi singkat. dampak teknologi ini semakin terasa dalam proses kreatif, mempercepat produktivitas dan membuka peluang baru bagi seniman dan kreator.


## Langkah 3 — Tokenisasi
Tokenisasi memecah teks menjadi satuan kata (token). Di sini digunakan *regular expression* `[a-z0-9]+`: setiap rangkaian huruf/angka dianggap satu token, sedangkan tanda baca menjadi pemisah.

In [5]:
import re

def tokenize(teks):
    return re.findall(r"[a-z0-9]+", teks.lower())

token_contoh = tokenize(case_folding(contoh))
print(len(token_contoh), "token")
print(token_contoh)

89 token
['peran', 'ai', 'generatif', 'dalam', 'industri', 'kreatif', 'kemajuan', 'teknologi', 'kecerdasan', 'buatan', 'ai', 'terus', 'berkembang', 'pesat', 'dan', 'memengaruhi', 'berbagai', 'sektor', 'industri', 'termasuk', 'industri', 'kreatif', 'ai', 'generatif', 'yang', 'menggunakan', 'algoritma', 'untuk', 'menciptakan', 'konten', 'baru', 'berdasarkan', 'data', 'yang', 'ada', 'telah', 'mengubah', 'cara', 'manusia', 'berinteraksi', 'dengan', 'seni', 'musik', 'desain', 'dan', 'bahkan', 'film', 'alat', 'alat', 'seperti', 'chatgpt', 'dan', 'dall', 'e', 'dari', 'openai', 'adalah', 'contoh', 'nyata', 'dari', 'ai', 'generatif', 'yang', 'mampu', 'menghasilkan', 'teks', 'dan', 'gambar', 'berdasarkan', 'deskripsi', 'singkat', 'dampak', 'teknologi', 'ini', 'semakin', 'terasa', 'dalam', 'proses', 'kreatif', 'mempercepat', 'produktivitas', 'dan', 'membuka', 'peluang', 'baru', 'bagi', 'seniman', 'dan', 'kreator']


In [6]:
# Bandingkan dengan tokenisasi naif menggunakan split() — tanda baca masih menempel
print(case_folding(contoh).split()[:12])

['peran', 'ai', 'generatif', 'dalam', 'industri', 'kreatif.', 'kemajuan', 'teknologi', 'kecerdasan', 'buatan', '(ai)', 'terus']


In [7]:
%pip install Sastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Langkah 4 — Penghapusan stopword
*Stopword* adalah kata yang sangat sering muncul tetapi hampir tidak membawa makna topik, misalnya *yang, dan, di, untuk*. Sastrawi menyediakan daftar stopword bahasa Indonesia.

In [8]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
print("Jumlah stopword:", len(STOPWORDS))
print(sorted(STOPWORDS)[:40])

Jumlah stopword: 123
['ada', 'adalah', 'agak', 'agar', 'akan', 'amat', 'anda', 'antara', 'anu', 'apakah', 'apalagi', 'atau', 'bagaimanapun', 'bagi', 'bahwa', 'begitu', 'belum', 'bisa', 'boleh', 'dahulu', 'dalam', 'dan', 'dapat', 'dari', 'daripada', 'demi', 'demikian', 'dengan', 'di', 'dia', 'dimana', 'dll', 'dsb', 'dst', 'dua', 'dulunya', 'guna', 'hal', 'hanya', 'harus']


In [9]:
def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

tanpa_sw = remove_stopwords(token_contoh)
print(len(token_contoh), "->", len(tanpa_sw), "token")
print(tanpa_sw)

89 -> 68 token
['peran', 'ai', 'generatif', 'industri', 'kreatif', 'kemajuan', 'teknologi', 'kecerdasan', 'buatan', 'ai', 'terus', 'berkembang', 'pesat', 'memengaruhi', 'berbagai', 'sektor', 'industri', 'termasuk', 'industri', 'kreatif', 'ai', 'generatif', 'menggunakan', 'algoritma', 'menciptakan', 'konten', 'baru', 'berdasarkan', 'data', 'mengubah', 'cara', 'manusia', 'berinteraksi', 'seni', 'musik', 'desain', 'bahkan', 'film', 'alat', 'alat', 'chatgpt', 'dall', 'e', 'openai', 'contoh', 'nyata', 'ai', 'generatif', 'mampu', 'menghasilkan', 'teks', 'gambar', 'berdasarkan', 'deskripsi', 'singkat', 'dampak', 'teknologi', 'semakin', 'terasa', 'proses', 'kreatif', 'mempercepat', 'produktivitas', 'membuka', 'peluang', 'baru', 'seniman', 'kreator']


## Langkah 5 — Stemming
*Stemming* mengubah kata berimbuhan menjadi kata dasar. Bahasa Indonesia kaya imbuhan (awalan, akhiran, sisipan, konfiks), sehingga Sastrawi memakai algoritma berbasis kamus kata dasar (Nazief–Adriani yang diperluas).

In [10]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
for kata in ["pembelajaran", "pertanian", "petani", "pencarian", "keamanan",
             "memantau", "berolahraga", "perekonomian", "mahasiswa"]:
    print(f"{kata:15s} -> {stemmer.stem(kata)}")

pembelajaran    -> ajar
pertanian       -> tani
petani          -> tani
pencarian       -> cari
keamanan        -> aman
memantau        -> pantau
berolahraga     -> olahraga
perekonomian    -> ekonomi
mahasiswa       -> mahasiswa


Stemming Sastrawi relatif lambat karena setiap kata dicocokkan ke kamus. Karena kata yang sama muncul berulang kali, hasilnya disimpan dalam *cache* (dictionary) agar setiap kata unik hanya di-stem sekali.

In [11]:
_cache = {}

def stem(token):
    if token not in _cache:
        _cache[token] = stemmer.stem(token)
    return _cache[token]

print([stem(t) for t in tanpa_sw])

['peran', 'ai', 'generatif', 'industri', 'kreatif', 'maju', 'teknologi', 'cerdas', 'buat', 'ai', 'terus', 'kembang', 'pesat', 'pengaruh', 'bagai', 'sektor', 'industri', 'masuk', 'industri', 'kreatif', 'ai', 'generatif', 'guna', 'algoritma', 'cipta', 'konten', 'baru', 'dasar', 'data', 'ubah', 'cara', 'manusia', 'interaksi', 'seni', 'musik', 'desain', 'bahkan', 'film', 'alat', 'alat', 'chatgpt', 'dall', 'e', 'openai', 'contoh', 'nyata', 'ai', 'generatif', 'mampu', 'hasil', 'teks', 'gambar', 'dasar', 'deskripsi', 'singkat', 'dampak', 'teknologi', 'makin', 'asa', 'proses', 'kreatif', 'cepat', 'produktivitas', 'buka', 'peluang', 'baru', 'seniman', 'kreator']


## Langkah 6 — Menggabungkan menjadi satu fungsi `preprocess`

In [12]:
def preprocess(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

korpus["tokens"] = korpus["teks"].apply(preprocess)
queries["tokens"] = queries["query"].apply(preprocess)  # pipeline konsisten
display(korpus[["id_doc", "judul", "tokens"]].head(), queries[["query_id", "tokens"]].head())


,id_doc,judul,tokens
0,D01,Pertanian,"[tani, tani, cakup, giat, budidaya, tanam, terna, hasil, pangan, bahan, baku, produk, praktik, tani, modern, gabung,..."
1,D02,Peran AI Generatif dalam Industri Kreatif,"[peran, ai, generatif, industri, kreatif, maju, teknologi, cerdas, buat, ai, terus, kembang, pesat, pengaruh, bagai,..."
2,D03,Peta,"[peta, makna, peta, gantung, konteks, 4, a, ranah, kartografi, definisi, awal, fokus, representasi, bumi, cetak, ata..."
3,D04,Branding Agency,"[branding, agency, branding, agency, usaha, milik, strategi, bawa, bisnis, kembang, arah, ingin, branding, agency, b..."
4,D05,Machine Learning,"[machine, learning, machine, learning, ajar, algoritma, bangun, pola, model, data, komputer, buat, prediksi, putus, ..."


,query_id,tokens
0,Q01,"[teknologi, cerdas, buat, hasil, konten, kreatif]"
1,Q02,"[algoritma, machine, learning, prediksi, data]"
2,Q03,"[sebab, dampak, bakar, hutan, lahan]"
3,Q04,"[manfaat, energi, matahari, sumber, energi, alternatif]"
4,Q05,"[sistem, operasi, kelola, perangkat, keras, komputer]"


## Langkah 7 — Mengukur pengaruh tiap tahap

In [13]:
tahap = {
    "Tokenisasi saja": dict(use_stopword=False, use_stemming=False),
    "+ Stopword removal": dict(use_stopword=True, use_stemming=False),
    "+ Stemming (tanpa stopword removal)": dict(use_stopword=False, use_stemming=True),
    "+ Stopword removal + Stemming": dict(use_stopword=True, use_stemming=True),
}
baris = []
for nama, opsi in tahap.items():
    semua = [t for teks in korpus["teks"] for t in preprocess(teks, **opsi)]
    baris.append({"Tahap": nama, "Jumlah token": len(semua), "Ukuran kosakata": len(set(semua))})
pd.DataFrame(baris)

,Tahap,Jumlah token,Ukuran kosakata
0,Tokenisasi saja,4462,1651
1,+ Stopword removal,3387,1574
2,+ Stemming (tanpa stopword removal),4462,1343
3,+ Stopword removal + Stemming,3387,1285


In [14]:
from collections import Counter

frekuensi = Counter(t for tokens in korpus["tokens"] for t in tokens)
pd.DataFrame(frekuensi.most_common(15), columns=["term", "frekuensi"])

,term,frekuensi
0,jadi,30
1,apple,30
2,guna,26
3,indonesia,26
4,usaha,24
5,data,22
6,tahun,22
7,komputer,22
8,hasil,17
9,teknologi,17


## Langkah 8 — Menyimpan fungsi untuk step berikutnya
Fungsi yang sama sudah disediakan di `stki_utils.py` agar notebook step berikutnya cukup mengimpornya. Sel berikut memastikan hasil fungsi Anda identik dengan versi di `stki_utils.py`.

In [16]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import stki_utils
cocok = all(preprocess(t) == stki_utils.preprocess(t) for t in korpus["teks"])
print("Hasil identik dengan stki_utils.preprocess:", cocok)
assert cocok


Hasil identik dengan stki_utils.preprocess: True


## Jawaban tugas Modul 1

1. **Overstemming / understemming**: tiga kasus nyata pada korpus diperiksa otomatis pada sel berikut. *Overstemming* berisiko menyatukan kata yang berbeda makna (false positive); *understemming* membuat varian kata tidak bertemu di indeks (false negative). Hasil Sastrawi bisa berbeda menurut versi pustaka.
2. **Stopword domain**: uji `dapat` dan `membantu` (bila tidak termasuk stopword bawaan). Lihat perbandingan kosakata secara langsung, dan jangan mengklaim pengurangan jika katanya sudah hilang dari stopword bawaan.
3. **Kata ulang**: tokenizer baru mempertahankan bentuk `anak-anak` / `sayur-mayur` sebagai satu token; diikuti uji otomatis. Ini perubahan untuk tugas saja, **bukan** penggantian pipeline resmi pada `stki_utils.py` agar konsisten dengan notebook selanjutnya.
4. **Analisis tahap terbesar**: lihat hasil perhitungan otomatis yang mencetak nama tahap serta alasan singkat, tanpa mengarang angka.


In [ ]:
# 1. Temukan contoh kasus stemming dari token yang MEMANG muncul di korpus
from collections import defaultdict, Counter
semua_asli = [t for x in korpus["teks"] for t in tokenize(x)]
unik_asli = sorted(set(semua_asli))
stem_group = defaultdict(list)
for t in unik_asli:
    stem_group[stem(t)].append(t)
kelompok = [(akar, bentuk) for akar, bentuk in stem_group.items() if len(bentuk) >= 2]
kelompok.sort(key=lambda pair: (-len(pair[1]), pair[0]))
print("Pasangan/kelompok bentuk berbeda yang dinormalisasi ke akar sama (kandidat analisis):")
for akar, bentuk in kelompok[:15]:
    print(f"  {akar:18s}: {', '.join(bentuk[:8])}")
print("Catatan: kelompok ini contoh penyamaan bentuk, bukan otomatis kesalahan overstemming.")

# Temukan contoh yang lebih mungkin bermasalah untuk diperiksa secara manual.
# Understemming: bentuk turunan belum menyatu dengan bentuk dasar yang hadir di korpus.
contoh_uji = ["menghasilkan", "pengelolaan", "pembelajaran", "mengembangkan", "penyimpanan", "kebakaran", "pertanian", "berkembang"]
print("\nAudit minimal 3 bentuk dari korpus (interpretasikan hasil):")
for kata in [x for x in contoh_uji if x in unik_asli][:8]:
    print(f"  {kata:18s} -> {stem(kata)}")
print("Pilih minimal tiga dari audit di atas dan tulis apakah betul ada over/understemming;")
print("jika tidak ditemukan, nyatakan hasil apa adanya. Jangan menamai stemming normal sebagai kesalahan.")

# 2. Stopword domain (pastikan kata tambahan belum termasuk daftar dasar)
tambahan = {"dapat", "membantu"}
stop_domain = STOPWORDS | tambahan
token_awal = [t for x in korpus["teks"] for t in tokenize(x)]
vocab_sebelum = set(remove_stopwords(token_awal))
vocab_sesudah = set(remove_stopwords(token_awal, stop_domain))
print("\nStopword tambahan yang baru:", sorted(tambahan - STOPWORDS))
print(f"Kosakata setelah stopword bawaan: {len(vocab_sebelum)}")
print(f"Kosakata setelah stopword domain: {len(vocab_sesudah)}")
print(f"Selisih kosakata: {len(vocab_sebelum)-len(vocab_sesudah)}")

# 3. Kata ulang sebagai satu token
def tokenize_kata_ulang(teks):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", teks.lower())
uji = "Anak-anak bermain bersama, sayur-mayur dijual di pasar."
print("\nTokenizer standar :", tokenize(uji))
print("Tokenizer kata ulang:", tokenize_kata_ulang(uji))
assert "anak-anak" in tokenize_kata_ulang(uji)
assert "sayur-mayur" in tokenize_kata_ulang(uji)

# 4. Ukuran kosakata: tampilkan hasil seluruh tahap pada data asli
tabel_tahap = pd.DataFrame(baris)
print("\nPerbandingan tahap:")
display(tabel_tahap)
terbesar = tabel_tahap.loc[tabel_tahap["Ukuran kosakata"].idxmin(), "Tahap"]
print("Analisis: konfigurasi dengan kosakata terkecil adalah", terbesar,
      ". Stopword removal membuang kata umum; stemming menyatukan sejumlah variasi berimbuhan menjadi akar yang sama. ",
      "Perubahan persisnya bergantung pada isi 50 dokumen dan daftar stopword.")
